# The Knapsack Problem (a variant)

## 1. Problem description

At a flea market in Rome, you spot $n$ objects (old pictures, a vessel, rusty medals, etc.) that you could re-sell in your antique shop for about double the price. 

You want these objects to pay for your flight ticket to Rome, which cost $C$.

Also, your backpack can carry all of them, but you don't want it heavy, so you want to buy the objects that will load your backpack as little as possible.

Each object $j=1,2,\ldots{},n$ has a price $p_j$ and a weight $w_j$.

Which objects should you buy?

## 2. Modeling

We have one variable $x_j$ for each $j=1,2\ldots{},n$. This is a binary variable:
$$ x_j = \begin{cases}
    1 & \text{ if you take object $j$ } \\
    0 & \text{ if you do not take object $j$ }
    \end{cases}
$$

Our objective is to minimize the toal weight:
$$ \min \quad \sum_{j=1}^n w_j x_j $$

The constraint is that the total revenue must be at least $C$:
$$ \sum_{j=1}^n p_j x_j \geq C $$

## 3. Implementation

### 3.1 A concrete model

In [ ]:
import pyomo.environ as pyo

In [ ]:
# Model parameters
n = 9
C = 70
w = [3, 2, 2, 4, 5, 4, 3, 1, 4]
p = [30, 24, 11, 35, 29, 8, 31, 18, 12]

In [ ]:
# Create a Concrete model
model = pyo.ConcreteModel()

In [ ]:
# Build the decision variables
model.x = pyo.Var(range(n), within=pyo.Binary)

$range(n)$ is a Python function which provides $\{0, 1, \ldots, n-1\}$.

In [ ]:
# Build the objective function
model.obj = pyo.Objective(expr=sum(w[i]*model.x[i] for i in range(n)))

$sum(w[i]*model.x[i]$ for $i$ in $range(n))$ is called a List Comprehension in Python. It is a short version of a for loop. 

The objective function can also be defined by a function (rule). The following lines can be used to replace the line above. 

In [ ]:
# An alternative way to define the objective function
def obj_value_rule(model):
    return sum(w[i] * model.x[i] for i in range(n))

model.obj = pyo.Objective(rule=obj_value_rule)

In [ ]:
# Build the constraint
model.cost_constraint = pyo.Constraint(expr=sum(p[i]*model.x[i] for i in range(n)) >= C)

Similarly, the constraint can also be defined by a rule. The code is omitted. 

In [ ]:
# Print the model
print()
model.pprint()

In [ ]:
# Solve the model through the GLPK solver
solver = pyo.SolverFactory('glpk')
solver.solve(model)

In [ ]:
# Print the optimzation results
print()
model.display()  # List of all optimization results
print()
print('Optimal value: ', pyo.value(model.obj))  # Print the value of model.obj (i.e., optimal objective value)

### 3.2 An abstract model

In [ ]:
import pyomo.environ as pyo

In [ ]:
# Create an Abstract model
model = pyo.AbstractModel()

An abstract model stores the basic model declarations, but does not construct the actual objects. At “creation time”, data is applied to the abstract declaration to create a concrete instance. Abstract modeling encourages generic modeling and model reuse.

In [ ]:
# Build parameters
model.n = pyo.Param(within=pyo.PositiveIntegers)
model.C = pyo.Param(within=pyo.NonNegativeReals)

model.n and model.C are "shells" without actual values.

In [ ]:
type(model.n)

In [ ]:
# Build parameters (using other parameters)
model.item_set = pyo.RangeSet(model.n)
model.w = pyo.Param(model.item_set, within=pyo.NonNegativeReals)
model.p = pyo.Param(model.item_set, within=pyo.NonNegativeReals)

$RangeSet(n)$ is a Pyomo function which provides $\{1, 2, \ldots, n\}$.

In [ ]:
# Build the decision variables
model.x = pyo.Var(model.item_set, within=pyo.Binary)

In [ ]:
# Build the objective function

def obj_value_rule(model):
    return sum(model.w[i] * model.x[i] for i in model.item_set)

model.obj = pyo.Objective(rule=obj_value_rule)

In an abstract model, the objective function and the constraints need to be built with functions (rules).

In [ ]:
# Build the constraint

def constraint_rule(model):
    return sum(model.p[i] * model.x[i] for i in model.item_set) >= model.C

model.cost_constraint = pyo.Constraint(rule=constraint_rule)

In [ ]:
# Load data
instance = model.create_instance('Knapsack.dat')

A file named 'Knapsack.dat' needs to be in the same folder. Or you can provide a path to the data file. A concreate instance is built in this step.

In [ ]:
# Print the model
instance.pprint()

In [ ]:
# Solve the model through the GLPK solver
solver = pyo.SolverFactory('glpk')
solver.solve(instance)

In [ ]:
# Print the results
instance.display()
print()
print('Optimal objective value: ', pyo.value(instance.obj))